# Bot Detection Methodologies: Literature Review and Research

## Objective
This notebook provides a comprehensive overview of bot detection methodologies for social media platforms. It serves as:
1. **Learning resource**: Understanding the theory and approaches
2. **Research documentation**: Methodology section for the research paper
3. **Implementation guide**: Foundation for our detection algorithms

## Table of Contents
1. Introduction to Bot Detection
2. Heuristic-Based Approaches
3. Machine Learning Approaches
4. Platform-Specific Considerations
5. Hybrid Approaches
6. Evaluation Metrics

## 1. Introduction to Bot Detection

### 1.1 What are Social Media Bots?

Social media bots are automated accounts that mimic human behavior on platforms like YouTube, Twitter, and Reddit. They can:
- Inflate follower counts artificially
- Generate spam comments
- Manipulate engagement metrics
- Spread misinformation

### 1.2 Why Bot Detection Matters for Influencer-Brand Mapping

**Data Quality**: Bots contaminate datasets with:
- Fake engagement signals
- Misleading follower counts
- Spam content that corrupts feature extraction

**Research Validity**: Accurate influencer-brand mapping requires:
- Authentic engagement metrics
- Real user interactions
- Genuine content patterns

**Key Citations**:
- Ferrara et al. (2016): "The Rise of Social Bots"
- Varol et al. (2017): "Online Human-Bot Interactions"
- Cresci et al. (2017): "The Paradigm-Shift of Social Spambots"

## 2. Heuristic-Based Approaches

### 2.1 Account-Level Features

**Profile Completeness**:
- **Indicator**: Bots often have incomplete profiles
- **Metrics**: Profile photo presence, bio length, verified status
- **Threshold**: Accounts with missing photos + short/no bio → suspicious

**Account Age**:
- **Indicator**: Recently created accounts with high activity are suspicious
- **Metrics**: Days since creation, activity rate
- **Threshold**: <30 days old + high posting rate → likely bot

**Follower/Following Ratio**:
- **Indicator**: Bots often follow many but have few followers
- **Formula**: `ratio = followers / (following + 1)`
- **Threshold**: ratio < 0.1 or ratio > 10 → suspicious

### 2.2 Behavioral Features

**Posting Frequency**:
- **Indicator**: Bots post at superhuman rates
- **Metrics**: Tweets per day, posts per hour
- **Threshold**: >50 posts/day consistently → likely bot

**Temporal Patterns**:
- **Indicator**: Bots show regular, non-human timing
- **Analysis**: Posting time distribution, inter-post intervals
- **Red flags**: Perfect 24/7 activity, exact hourly posts

**Engagement Rate**:
- **YouTube**: `engagement = (likes + comments) / views`
- **Twitter**: `engagement = (likes + retweets + replies) / followers`
- **Threshold**: Abnormally low (<0.001) or high (>0.5) → suspicious

### 2.3 Content-Level Features

**Content Diversity**:
- **Indicator**: Bots often post repetitive content
- **Metrics**: Text similarity, unique word ratio
- **Detection**: Cosine similarity between consecutive posts

**Spam Patterns**:
- URL density (links per post)
- Hashtag density
- Generic/template phrases
- Special character overuse

**Language Quality**:
- Grammar and spelling errors
- Coherence and context relevance
- Machine-generated text patterns

In [ ]:
# Example: Heuristic Score Function
import numpy as np

def calculate_bot_score_heuristic(account_features):
    """
    Calculate bot likelihood score using heuristics.
    
    Parameters:
    - account_features: dict with keys:
        - has_profile_photo: bool
        - bio_length: int
        - account_age_days: int
        - followers: int
        - following: int
        - posts_per_day: float
        - engagement_rate: float
    
    Returns:
    - bot_score: float [0-1], higher = more likely bot
    """
    score = 0.0
    
    # Profile completeness (20% weight)
    if not account_features.get('has_profile_photo', True):
        score += 0.1
    if account_features.get('bio_length', 100) < 20:
        score += 0.1
    
    # Account age (20% weight)
    age = account_features.get('account_age_days', 365)
    if age < 30:
        score += 0.2
    elif age < 90:
        score += 0.1
    
    # Follower ratio (20% weight)
    followers = account_features.get('followers', 100)
    following = account_features.get('following', 100)
    ratio = followers / (following + 1)
    if ratio < 0.1 or ratio > 10:
        score += 0.2
    
    # Posting frequency (20% weight)
    posts_per_day = account_features.get('posts_per_day', 5)
    if posts_per_day > 50:
        score += 0.2
    elif posts_per_day > 30:
        score += 0.1
    
    # Engagement rate (20% weight)
    engagement = account_features.get('engagement_rate', 0.05)
    if engagement < 0.001 or engagement > 0.5:
        score += 0.2
    
    return min(score, 1.0)

# Example usage
example_account = {
    'has_profile_photo': False,
    'bio_length': 10,
    'account_age_days': 15,
    'followers': 50,
    'following': 5000,
    'posts_per_day': 60,
    'engagement_rate': 0.0005
}

bot_score = calculate_bot_score_heuristic(example_account)
print(f"Bot Score: {bot_score:.2f}")
print(f"Classification: {'BOT' if bot_score > 0.5 else 'HUMAN'}")

## 3. Machine Learning Approaches

### 3.1 Feature Engineering

**Account Features** (10-15 features):
1. Profile completeness score
2. Account age (days)
3. Follower count (log scale)
4. Following count (log scale)
5. Follower/following ratio
6. Posts count
7. Verified status (binary)

**Behavioral Features** (15-20 features):
1. Posts per day (mean, std)
2. Inter-post time intervals (mean, std)
3. Posting hour distribution entropy
4. Posting day of week distribution
5. Activity consistency score

**Content Features** (10-15 features):
1. Average post length
2. Content diversity (unique words ratio)
3. URL density
4. Hashtag density
5. Mention density
6. Text similarity between posts (mean)
7. Sentiment distribution

**Engagement Features** (5-10 features):
1. Average likes per post
2. Average comments per post
3. Engagement rate
4. Engagement consistency (std)

### 3.2 Supervised Learning Models

**Random Forest**:
- **Pros**: Handles non-linear relationships, feature importance, robust to outliers
- **Cons**: Requires labeled data, can overfit with many trees
- **Use case**: When we have ground truth labels for bots

**Logistic Regression**:
- **Pros**: Interpretable, fast, works well with binary classification
- **Cons**: Assumes linear relationships
- **Use case**: Baseline model, when interpretability is key

**Gradient Boosting (XGBoost)**:
- **Pros**: High accuracy, handles imbalanced data
- **Cons**: Slower training, requires tuning
- **Use case**: When maximizing accuracy is priority

### 3.3 Unsupervised Learning Models

**Isolation Forest**:
- **Approach**: Anomaly detection based on feature isolation
- **Use case**: When labeled data is scarce
- **Assumption**: Bots are anomalies in feature space

**One-Class SVM**:
- **Approach**: Learn boundary of normal (human) behavior
- **Use case**: When we have many human examples, few bot examples

**Clustering (K-Means, DBSCAN)**:
- **Approach**: Group similar accounts, identify bot clusters
- **Use case**: Exploratory analysis, pattern discovery

In [ ]:
# Example: Feature Engineering Pipeline
import pandas as pd
from datetime import datetime, timedelta

def engineer_bot_detection_features(account_data, posts_data):
    """
    Engineer features for bot detection ML model.
    
    Parameters:
    - account_data: DataFrame with account metadata
    - posts_data: DataFrame with post history
    
    Returns:
    - features_df: DataFrame with engineered features
    """
    features = {}
    
    # Account features
    features['account_age_days'] = (datetime.now() - pd.to_datetime(account_data['created_at'])).days
    features['follower_count_log'] = np.log10(account_data['followers'] + 1)
    features['following_count_log'] = np.log10(account_data['following'] + 1)
    features['follower_ratio'] = account_data['followers'] / (account_data['following'] + 1)
    features['has_profile_photo'] = int(account_data.get('profile_photo', '') != '')
    features['bio_length'] = len(account_data.get('bio', ''))
    
    # Behavioral features
    if len(posts_data) > 0:
        posts_data['timestamp'] = pd.to_datetime(posts_data['created_at'])
        posts_data = posts_data.sort_values('timestamp')
        
        # Posting frequency
        time_range = (posts_data['timestamp'].max() - posts_data['timestamp'].min()).days + 1
        features['posts_per_day'] = len(posts_data) / time_range
        
        # Inter-post intervals
        intervals = posts_data['timestamp'].diff().dt.total_seconds() / 3600  # hours
        features['interpost_mean_hours'] = intervals.mean()
        features['interpost_std_hours'] = intervals.std()
        
        # Temporal patterns
        posts_data['hour'] = posts_data['timestamp'].dt.hour
        hour_dist = posts_data['hour'].value_counts(normalize=True)
        features['posting_hour_entropy'] = -(hour_dist * np.log(hour_dist + 1e-10)).sum()
        
        # Content features
        posts_data['text_length'] = posts_data['text'].astype(str).str.len()
        features['avg_post_length'] = posts_data['text_length'].mean()
        features['std_post_length'] = posts_data['text_length'].std()
        
        # URL and hashtag density
        posts_data['url_count'] = posts_data['text'].str.count(r'http[s]?://')
        posts_data['hashtag_count'] = posts_data['text'].str.count(r'#\w+')
        features['url_density'] = posts_data['url_count'].mean()
        features['hashtag_density'] = posts_data['hashtag_count'].mean()
        
        # Engagement features
        features['avg_likes'] = posts_data['likes'].mean()
        features['avg_comments'] = posts_data.get('comments', posts_data.get('replies', 0)).mean()
        features['engagement_rate'] = (posts_data['likes'].sum() + 
                                      posts_data.get('comments', posts_data.get('replies', 0)).sum()) / \
                                     (account_data['followers'] + 1)
    else:
        # Default values if no posts
        features.update({
            'posts_per_day': 0,
            'interpost_mean_hours': 0,
            'interpost_std_hours': 0,
            'posting_hour_entropy': 0,
            'avg_post_length': 0,
            'std_post_length': 0,
            'url_density': 0,
            'hashtag_density': 0,
            'avg_likes': 0,
            'avg_comments': 0,
            'engagement_rate': 0
        })
    
    return pd.Series(features)

print("Feature engineering pipeline defined.")
print("This will extract ~20 features for ML model training.")

## 4. Platform-Specific Considerations

### 4.1 YouTube Bot Detection

**Channel-Level Bots**:
- **Sub4Sub networks**: Fake subscriber exchange
- **View bots**: Artificial view inflation
- **Indicators**:
  - Subscriber count >> avg views per video
  - Very low engagement (likes/views, comments/views)
  - Sudden subscriber spikes without content changes

**Comment-Level Bots**:
- **Generic comments**: "Great video!", "Nice!"
- **Spam links**: Promotional URLs
- **Detection**:
  - Text similarity clustering
  - Link pattern matching
  - Account age + comment frequency

**YouTube-Specific Metrics**:
```python
subscriber_view_ratio = total_views / (subscribers * total_videos + 1)
# Healthy range: 0.1 - 10
# < 0.1: likely fake subscribers
# > 10: likely view botting

engagement_score = (likes + comments * 5) / views
# Typical range: 0.01 - 0.05
# < 0.001: suspiciously low engagement
```

### 4.2 Twitter Bot Detection

**Follower Bots**:
- Purchased followers to inflate count
- Fake engagement networks

**Tweet Bots**:
- Automated tweeting
- Retweet/reply bots
- Spam/promotional bots

**Twitter-Specific Metrics**:
```python
bot_score_twitter = (
    0.3 * default_profile_indicator +  # Using default profile image
    0.2 * high_following_ratio +       # Following >> followers
    0.2 * tweet_frequency_score +      # Superhuman tweet rate
    0.15 * content_diversity_score +   # Repetitive content
    0.15 * engagement_anomaly_score    # Unusual engagement patterns
)
```

### 4.3 Reddit Bot Detection

**Account Bots**:
- Karma farming bots
- Repost bots
- Astroturfing accounts

**Indicators**:
- High posting frequency (>20 posts/day)
- Low subreddit diversity (posting in 1-2 subs only)
- Reposted content (exact duplicates)
- Account age vs. karma ratio

**Reddit-Specific Metrics**:
```python
karma_rate = total_karma / account_age_days
# > 1000/day consistently: suspicious

subreddit_diversity = unique_subreddits / total_posts
# < 0.1: likely bot (posting in same sub repeatedly)

content_originality = unique_posts / total_posts
# < 0.5: likely repost bot
```

## 5. Hybrid Approach (Recommended)

### 5.1 Two-Stage Detection Pipeline

**Stage 1: Heuristic Filtering** (Fast, High Recall)
- Apply rule-based thresholds
- Flag obvious bots (score > 0.7)
- Pass uncertain accounts (0.3 < score < 0.7) to ML stage
- Keep likely humans (score < 0.3)

**Stage 2: ML Classification** (Accurate, High Precision)
- Apply trained ML model to uncertain accounts
- Use ensemble of Random Forest + Logistic Regression
- Final classification based on probability threshold

### 5.2 Advantages of Hybrid Approach

1. **Efficiency**: Heuristics filter out obvious cases quickly
2. **Accuracy**: ML handles edge cases with higher precision
3. **Interpretability**: Heuristic rules explain why accounts are flagged
4. **Adaptability**: ML can learn new bot patterns

### 5.3 Implementation Strategy

```python
def hybrid_bot_detection(account):
    # Stage 1: Heuristic filtering
    heuristic_score = calculate_bot_score_heuristic(account)
    
    if heuristic_score > 0.7:
        return {'is_bot': True, 'confidence': 'high', 'method': 'heuristic'}
    elif heuristic_score < 0.3:
        return {'is_bot': False, 'confidence': 'high', 'method': 'heuristic'}
    else:
        # Stage 2: ML classification
        features = engineer_features(account)
        ml_probability = ml_model.predict_proba(features)[0][1]
        
        is_bot = ml_probability > 0.5
        confidence = 'high' if abs(ml_probability - 0.5) > 0.3 else 'medium'
        
        return {
            'is_bot': is_bot,
            'confidence': confidence,
            'method': 'ml',
            'probability': ml_probability
        }
```

## 6. Evaluation Metrics

### 6.1 Classification Metrics

**Confusion Matrix**:
```
                Predicted
              Bot    Human
Actual Bot    TP     FN
Actual Human  FP     TN
```

**Key Metrics**:
- **Precision**: TP / (TP + FP) - Of accounts we flagged, how many are actually bots?
- **Recall**: TP / (TP + FN) - Of all actual bots, how many did we catch?
- **F1 Score**: 2 × (Precision × Recall) / (Precision + Recall)
- **Accuracy**: (TP + TN) / Total

### 6.2 Which Metric to Prioritize?

**For Bot Detection**: **High Precision** is preferred
- **Why**: We don't want to remove legitimate users
- **Trade-off**: Accept some bots to avoid false positives
- **Target**: Precision > 0.9, Recall > 0.7

### 6.3 ROC Curve and AUC

- **ROC Curve**: Trade-off between True Positive Rate and False Positive Rate
- **AUC**: Area Under Curve - overall model quality
- **Target AUC**: > 0.85 for good bot detection

### 6.4 Business Metrics

**Data Quality Improvement**:
```python
quality_improvement = (
    engagement_rate_after_cleaning / engagement_rate_before_cleaning
)
# Expected: 1.2 - 2.0 (20-100% improvement)
```

**Bot Removal Rate**:
```python
bot_rate = bots_removed / total_accounts
# Typical: 5-15% for organic data
# Higher rates may indicate data quality issues
```

## 7. Research Paper - Methodology Section Template

### Bot Detection Methodology

**Abstract**:
"We employ a hybrid bot detection approach combining rule-based heuristics with machine learning classification. This two-stage pipeline achieves high precision (>0.9) while maintaining acceptable recall (>0.7), ensuring data quality for downstream influencer-brand mapping tasks."

**Detailed Methodology**:

1. **Heuristic Filtering Stage**:
   - We define platform-specific heuristics based on account metadata, behavioral patterns, and content characteristics
   - Accounts are scored on a 0-1 scale using weighted combination of indicators
   - High-confidence classifications (score < 0.3 or > 0.7) bypass ML stage

2. **Feature Engineering**:
   - Extract 20+ features across four categories:
     - Account features (7): Profile completeness, account age, follower metrics
     - Behavioral features (8): Posting frequency, temporal patterns, activity consistency
     - Content features (7): Text quality, diversity, spam indicators
     - Engagement features (4): Likes, comments, engagement rates

3. **Machine Learning Classification**:
   - Ensemble approach using Random Forest and Logistic Regression
   - Training on manually labeled dataset (stratified 80/20 split)
   - Threshold tuning to maximize precision (target: >0.9)

4. **Evaluation**:
   - Report precision, recall, F1, and AUC on held-out test set
   - Analyze feature importance for interpretability
   - Measure data quality improvement post-filtering

**Expected Results**:
- Bot detection precision: 0.90-0.95
- Bot detection recall: 0.70-0.85
- F1 Score: 0.79-0.89
- AUC: 0.85-0.92
- Bot removal rate: 8-12% of total accounts
- Engagement quality improvement: 30-50%

## 8. Implementation Checklist

### Phase 1: Heuristic Implementation
- [ ] Define platform-specific thresholds
- [ ] Implement scoring function
- [ ] Test on sample data
- [ ] Tune thresholds for optimal precision/recall

### Phase 2: Feature Engineering
- [ ] Extract account features
- [ ] Extract behavioral features
- [ ] Extract content features
- [ ] Extract engagement features
- [ ] Handle missing values
- [ ] Scale/normalize features

### Phase 3: ML Model Training
- [ ] Create labeled training set (manual or bootstrapped)
- [ ] Split train/validation/test sets
- [ ] Train multiple models (RF, LR, XGBoost)
- [ ] Tune hyperparameters
- [ ] Evaluate on test set
- [ ] Select best model or ensemble

### Phase 4: Integration
- [ ] Combine heuristic + ML pipeline
- [ ] Process all platform data
- [ ] Generate cleaned datasets
- [ ] Document bot removal statistics
- [ ] Validate data quality improvement

## 9. Key Takeaways

1. **Hybrid approach** (heuristics + ML) provides best balance of speed and accuracy
2. **Platform-specific** features are crucial - YouTube, Twitter, and Reddit have different bot patterns
3. **High precision** is more important than recall to avoid removing legitimate users
4. **Feature engineering** is critical - behavioral and temporal features often outperform account metadata
5. **Continuous monitoring** needed as bot strategies evolve

## Next Steps

Proceed to implementation notebooks:
1. YouTube bot detection (heuristics + ML)
2. Twitter bot detection (heuristics + ML)
3. Reddit bot detection (heuristics + ML)
4. Integration and evaluation

In [ ]:
print("\n" + "="*60)
print("Bot Detection Research Complete!")
print("="*60)
print("\nKey Concepts Covered:")
print("✓ Heuristic-based detection approaches")
print("✓ Machine learning classification methods")
print("✓ Platform-specific considerations")
print("✓ Hybrid detection pipeline")
print("✓ Evaluation metrics and methodology")
print("\nReady to implement bot detection algorithms!")